# OpenWebUI + Ollama on Kaggle

这是一份全新 notebook（不修改你原有文件），用于在 Kaggle 里运行 OpenWebUI + Ollama。

按顺序执行下面代码单元：
1. 安装依赖
2. 启动 Ollama
3. 拉取模型并测试
4. 启动 OpenWebUI
5. 用 ngrok 暴露 OpenWebUI 公网访问地址

注意：Kaggle Session 重启后需要重新运行。

In [ ]:
# 1) 安装依赖
import subprocess


def run(cmd: str):
    print(f"\n>>> {cmd}")
    subprocess.run(cmd, shell=True, check=True)

run("apt-get update -y")
run("apt-get install -y curl wget git zstd")
run("curl -fsSL https://ollama.com/install.sh | sh")
run("python -m pip -q install --upgrade pip")
run("python -m pip -q install open-webui")

print("\n依赖安装完成")

In [ ]:
# 2) 启动 Ollama 服务
import os
import subprocess
import time
from pathlib import Path

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
log_dir = Path("/kaggle/working/logs")
log_dir.mkdir(parents=True, exist_ok=True)

ollama_log = open(log_dir / "ollama.log", "w")
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

# 等待服务就绪
time.sleep(5)
print(f"Ollama PID: {ollama_proc.pid}")
print("Ollama endpoint: http://127.0.0.1:11434")
print("日志文件: /kaggle/working/logs/ollama.log")

In [ ]:
# 3) 拉取模型并做最小测试
import subprocess

MODEL_NAME = "qwen2.5:0.5b"  # 可改成其他模型，例如 llama3.2:1b

subprocess.run(f"ollama pull {MODEL_NAME}", shell=True, check=True)
print("模型拉取完成:", MODEL_NAME)

subprocess.run(
    f"ollama run {MODEL_NAME} \"Reply with: ollama is ready\"",
    shell=True,
    check=True,
)

In [ ]:
# 4) 启动 OpenWebUI
import os
import sys
import time
import shutil
import subprocess
import urllib.request
from pathlib import Path

os.environ["OLLAMA_BASE_URL"] = "http://127.0.0.1:11434"
os.environ["WEBUI_AUTH"] = "False"  # 演示时关闭登录
os.environ["DATA_DIR"] = "/kaggle/working/openwebui-data"  # 避免写入受限目录

log_dir = Path("/kaggle/working/logs")
log_dir.mkdir(parents=True, exist_ok=True)
Path(os.environ["DATA_DIR"]).mkdir(parents=True, exist_ok=True)

# 优先使用 open-webui 可执行入口（python -m open_webui 在该版本下不可直接执行）
openwebui_bin = shutil.which("open-webui")
if openwebui_bin is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open-webui"], check=True)
    openwebui_bin = shutil.which("open-webui")
    if openwebui_bin is None:
        raise RuntimeError("open-webui executable not found after installation")

openwebui_log_path = log_dir / "openwebui.log"
webui_log = open(openwebui_log_path, "w")
webui_proc = subprocess.Popen(
    [openwebui_bin, "serve", "--host", "0.0.0.0", "--port", "8080"],
    stdout=webui_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

# 首次启动可能较慢（数据库迁移/初始化），最多等待 120 秒
ready = False
for _ in range(120):
    if webui_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=2) as resp:
            if resp.status < 500:
                ready = True
                break
    except Exception:
        pass
    try:
        with urllib.request.urlopen("http://127.0.0.1:8080", timeout=2) as resp:
            if resp.status < 500:
                ready = True
                break
    except Exception:
        pass
    time.sleep(1)

print(f"OpenWebUI PID: {webui_proc.pid}")
print("本地地址: http://127.0.0.1:8080")
print("日志文件: /kaggle/working/logs/openwebui.log")
print("OpenWebUI ready:", ready)

if webui_proc.poll() is not None:
    print("OpenWebUI 进程已退出，退出码:", webui_proc.returncode)

if not ready:
    try:
        tail = openwebui_log_path.read_text(errors="ignore").splitlines()[-80:]
        print("\n===== openwebui.log tail =====")
        print("\n".join(tail))
    except Exception:
        pass
    raise RuntimeError("OpenWebUI failed to become ready within 120s. See log tail above.")

In [ ]:
# 5) 启动 ngrok 暴露 OpenWebUI 公网访问地址
import sys
import time
import subprocess
import urllib.request

# 按你的要求硬编码 token
Ngrok_token = "21JxiosD62PLsE9BJL6AQqZRqkF_7fw6tKfScfb8aureupPzE"
OPENWEBUI_PORT = 8080

# 先安装 pyngrok，再导入
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok==6.1.0"], check=True)
from pyngrok import ngrok

# 等待 OpenWebUI 就绪（给足时间）
ready = False
for _ in range(120):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{OPENWEBUI_PORT}/health", timeout=2) as resp:
            if resp.status < 500:
                ready = True
                break
    except Exception:
        pass
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{OPENWEBUI_PORT}", timeout=2) as resp:
            if resp.status < 500:
                ready = True
                break
    except Exception:
        pass
    time.sleep(1)

if not ready:
    raise RuntimeError("OpenWebUI is not ready on port 8080. Run cell 4 to see detailed log tail.")

# 建立 ngrok 隧道
ngrok.set_auth_token(Ngrok_token)
ngrok.kill()  # 清理旧隧道，避免重复
public_tunnel = ngrok.connect(OPENWEBUI_PORT)

print("Ngrok OpenWebUI URL:")
print(public_tunnel.public_url)

In [ ]:
# 6) 故障排查：查看日志与端口
!echo "===== ollama.log ====="
!tail -n 120 /kaggle/working/logs/ollama.log
!echo "===== openwebui.log ====="
!tail -n 120 /kaggle/working/logs/openwebui.log
!echo "===== listening ports (11434/8080) ====="
!ss -lntp | grep -E "11434|8080" || true